In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from utils.data import EchocardiographyDataCollection

# Some General Information About the Dataset

In [ ]:
# load the data and split it into a training and a validation dataset
C = EchocardiographyDataCollection(overwrite=False)
C_val = C.train_val_split(validation_fraction=0.3, seed=25)

In [ ]:
# generate some general information about the training and the validation dataset
C.summary(C_val)

In [ ]:
def time_positive(time, curve):
    time_total = 0
    l, r = 0, 0
    mask_positive = (curve > 0)
    
    while min(l, r) < len(mask_positive)-1:
        if mask_positive[r] and r < len(mask_positive)-1:
            r += 1
        else:
            if l < r:
                time_total += time[r-1] - time[l]
                l = r

            else:
                l += 1
                r += 1
    
    return time_total

def is_positive_for_long(ECD):
    data = ECD.time_normalised_data['Longitudinal Strain-Endo'][0]
    curves = data.iloc[:, :-1].to_numpy()
    time = data.iloc[:, -1].to_numpy()
    mask_positive = (curves > 0)
    
    positive_for_long = np.any([time_positive(time, curves[:, i]) > 1/3 for i in range(curves.shape[1])])
    
    return positive_for_long

long_candidates = C.filter_keys_for_condition(is_positive_for_long)

In [ ]:
n_sampling_points = []

for group, patient in long_candidates:
    ECD = C.collection[group][patient]
    data = ECD.data['Longitudinal Strain-Endo']
    n_sampling_points.append(len(data))


plt.hist(n_sampling_points, bins=30)
plt.title(f'Average number of sampling points in a recording: {np.mean(n_sampling_points):.3f}')

In [ ]:
long_segments = {}
long_segments_patients = {}
times = {}

for group, patient in long_candidates:
    if group == 'control':
        ECD = C.collection[group][patient]
        data = ECD.time_normalised_data['Longitudinal Strain-Endo'][0]
        curves = data.iloc[:, :-1]
        time = data.iloc[:, -1].to_numpy()
        
        seg_names = curves.columns
        curves = curves.to_numpy()
        
        mask_positive = (curves > 0)

        n_segments = 0
        
        for i, seg_name in enumerate(seg_names):
            
            time_pos_total = time_positive(time, curves[:, i])
            if not seg_name in times.keys():
                times[seg_name] = []
            times[seg_name].append(time_pos_total)
            
            if time_pos_total > 1/3:
                if not seg_name in long_segments.keys():
                    long_segments[seg_name] = 0
                
                long_segments[seg_name] += 1
                n_segments += 1

        if n_segments > 0:
            if not n_segments in long_segments_patients.keys():
                    long_segments_patients[n_segments] = []
                
            long_segments_patients[n_segments].append(patient)

for seg_name, n_controls in long_segments.items():
    print(f'{seg_name}:\navg. pos. time = {np.mean(times[seg_name]):.2f}\nmin/max = {np.min(times[seg_name]):.2f}/{np.max(times[seg_name]):.2f}\nn controls = {n_controls}\n')

for n_segments, patients in long_segments_patients.items():
    print(f'{n_segments} segments too positive: patients (n = {len(patients)}) = {patients}')

# Visualisations Based on Strain Data

In [ ]:
C_lin = C.transform_data_to_common_grid(interpolation='linear', n_time_points=300)

seg = 1
seg_name = '02-free-wall-med'

colours = {'control': 'tab:blue',
           'disease': 'tab:red'
          }

curves_list = []
time_list = []
time_PS_list = []
groups = []

for (group, patient) in C_lin.keys_to_patients:
    data = C_lin.collection[group][patient].time_normalised_data['Longitudinal Strain-Endo'][0]
    time = data.iloc[:, -1].to_numpy()
    time_list.append(time)
    curves = data.iloc[:, :-1].to_numpy()
    curves_list.append(curves)
    
    index_PS = np.argmin(curves, axis=0)
    assert index_PS.shape == (6,)
    time_PS_list.append(time[index_PS])

    groups.append(group)

mean_t_PS = np.mean(np.stack(time_PS_list), axis=0)
assert mean_t_PS.shape == (6,)


curves_list = np.array(curves_list)
time_list = np.array(time_list)
time_PS_list = np.array(time_PS_list)
groups = np.array(groups)

curves_by_group = {'control': [], 'disease': []}
common_time = None

for group in ['control', 'disease']:
    mask_group = groups == group
    fig, axs = plt.subplots(1, 2, figsize=(7.5, 4), dpi=150, sharex=True, sharey=True)

    for curves, time, time_PS in zip(curves_list[mask_group], time_list[mask_group], time_PS_list[mask_group]):
        if common_time is None:
            common_time = time
        aligned_time = np.interp(x=time, xp=[0.0, time_PS[seg], 1.0], fp=[0.0, mean_t_PS[seg], 1.0])
        axs[0].plot(aligned_time, curves[:, seg], color=colours[group], alpha=0.4)
        curves_by_group[group].append(np.interp(x=time, xp=aligned_time, fp=curves[:, seg]))
    
    curves_by_group[group] = np.stack(curves_by_group[group])
    mean_curve = np.mean(curves_by_group[group], axis=0)
    upper = np.quantile(curves_by_group[group], 0.75, axis=0)
    lower = np.quantile(curves_by_group[group], 0.25, axis=0)
    
    axs[1].plot(common_time, mean_curve, color=colours[group], ls='--')
    axs[1].fill_between(common_time, upper, lower, color=colours[group], alpha=0.3, label='interquartile range')

    
    axs[0].set_title(f'All individuals')
    axs[1].set_title(f'Average curve')
    
    axs[1].legend()
    
    for ax in axs:
        ax.grid()
        ax.set_ylim([-60, 25])
        ax.set_xlabel(r'Relative time $t \in [0,1]$')
    
    axs[0].set_ylabel(f'$LS^p_{{{seg+1}}}(\gamma_p(t))$, aligned longitudinal strain [%]')
    
    fig.suptitle(f'Peak-strain-aligned longitudinal strain curves for {group} group\nin {seg_name} segment', fontweight='bold')
    fig.tight_layout()

In [ ]:
means = {}
common_grid = None

for group in C.collection:
    
    fig, axs = plt.subplots(6, 2, figsize=(7.5, 20), dpi=150, sharex=True, sharey=True)
    fig.suptitle(f'Longitudinal Strain data for {group} group', fontweight='bold')
    
    if not group in means.keys():
        means[group] = {}

    display_information = True
    
    for patient in C.collection[group]:
        display_information = True
        
        ECD = C_lin.collection[group][patient]        
        
        data_complete = ECD.time_normalised_data['Longitudinal Strain-Endo'][0]
        LS_segments = data_complete.iloc[:, :-1]
        time = data_complete.iloc[:, -1]

        if common_grid is None:
            common_grid = time
            
        for j, segment in enumerate(sorted(LS_segments.columns)):
            if not segment in means[group].keys():
                means[group][segment] = []
            
            means[group][segment].append(LS_segments[segment])
                
            ax = axs[j, 0]
            
            if display_information:
                ax.plot(time, 
                    LS_segments[segment], 
                    color=colours[group], 
                    alpha=0.4
                   )

                ax.set_title('All individuals in segment\n' + segment)
                if j == 5:
                    ax.set_xlabel(r'Relative time $t \in [0, 1]$')
                ax.set_ylabel('Longitudinal Strain [%]')
                ax.set_ylim([-60.0, 25.0])
                ax.grid(True)

            else:
                ax.plot(time, 
                        LS_segments[segment],
                        color=colours[group], 
                        alpha=0.4
                       )

        display_information = False

    
    for j, segment in enumerate(means[group].keys()):
        ax = axs[j, 1]
        
        mean_curve = np.mean(np.stack(means[group][segment]), axis=0)
        lower_perc = np.percentile(np.stack(means[group][segment]), 25, axis=0)
        upper_perc = np.percentile(np.stack(means[group][segment]), 75, axis=0)
        
        ax.plot(common_grid, mean_curve, ls='--', color=colours[group])
        ax.fill_between(common_grid, lower_perc, upper_perc, alpha=0.3, color=colours[group], label='interquartile range')
        

        ax.set_ylim([-60.0, 25.0])
        ax.legend()
        ax.grid(True)
        ax.set_title('Average curves in segment\n' + segment)
        if j == 5:
            ax.set_xlabel(r'Relative time $t \in [0, 1]$')
            
    fig.tight_layout(rect=[0, 0.03, 1, 0.97])

---
# Classification of deformation patterns

In [ ]:
from utils.mast_classification import classify_deformation_pattern, MAST_TYPE_COLORS

classifications = {}
types = MAST_TYPE_COLORS.keys()

for group, patient in C.keys_to_patients:
    ECD = C.collection[group][patient]
    location = ECD.location
    key = group + '\n(' + location + ')'
    
    if not key in classifications.keys():
        classifications[key] = {t: 0 for t in types}

    if not 'disease total' in classifications.keys():
        classifications['disease total'] = {t: 0 for t in types}
        
    res = classify_deformation_pattern(ECD)
    classifications[key][res] += 1
    if group == 'disease':
        classifications['disease total'][res] += 1


plt.figure(figsize=(7, 4), dpi=200)
groups = list(classifications.keys())
xs = np.arange(len(groups))
print_labels = True

for x, group in zip(xs, groups):
    for i, t in enumerate(types):
        if print_labels:
            plt.bar(x - 0.2 + 0.2*i, classifications[group][t], width=0.2, label=t, color=MAST_TYPE_COLORS[t], edgecolor='black')
        else:
            plt.bar(x - 0.2 + 0.2*i, classifications[group][t], width=0.2, color=MAST_TYPE_COLORS[t], edgecolor='black')
            
    print_labels = False

plt.axvline(1.5, ls='--', color='black')

plt.title('Classification into three types based on the rule from Mast et al. (2016)')
plt.xticks(xs, classifications.keys())
plt.legend()
plt.tight_layout()

In [ ]:
loc_class = {'Type-I': 0,
             'Type-II': 1,
             'Type-III': 2
            }

means = {}
common_grid = None

for group in C.collection:
    
    fig, axs = plt.subplots(6, 2, figsize=(7.5, 20), dpi=300, sharex=True, sharey=True)
    fig.suptitle(f'Longitudinal Strain data for {group} group', fontweight='bold')
    
    classes_plotted = []
    if not group in means.keys():
        means[group] = {}
    
    for patient in C.collection[group]:
        display_information = True
        
        ECD = C_lin.collection[group][patient]        
        
        mast_class = classify_deformation_pattern(ECD)
        if not mast_class in classes_plotted:
            classes_plotted.append(mast_class)
        else:
            display_information = False
        
        data_complete = ECD.time_normalised_data['Longitudinal Strain-Endo'][0]
        LS_segments = data_complete.iloc[:, :-1]
        time = data_complete.iloc[:, -1]

        if common_grid is None:
            common_grid = time
            
        for j, segment in enumerate(sorted(LS_segments.columns)):
            if not segment in means[group].keys():
                means[group][segment] = {}
            if not mast_class in means[group][segment].keys():
                means[group][segment][mast_class] = []

            means[group][segment][mast_class].append(LS_segments[segment])
                
            ax = axs[j, 0]
            
            if display_information:
                ax.plot(time, 
                    LS_segments[segment], 
                    label=mast_class, 
                    color=MAST_TYPE_COLORS[mast_class], 
                    alpha=0.4
                   )

                ax.set_title('All individuals in segment\n' + segment)
                ax.legend()
                if j == 5:
                    ax.set_xlabel(r'Relative time $t/T_i$')
                ax.set_ylabel('Longitudinal Strain [%]')
                ax.set_ylim([-60.0, 25.0])
                ax.grid(True)

            else:
                ax.plot(time, 
                        LS_segments[segment],
                        color=MAST_TYPE_COLORS[mast_class], 
                        alpha=0.4
                       )

        if display_information:
            display_information = False
    
    for j, segment in enumerate(means[group].keys()):
        ax = axs[j, 1]
        
        for t in sorted(means[group][segment].keys()):
            mean_curve = np.mean(np.stack(means[group][segment][t]), axis=0)
            ax.plot(common_grid, mean_curve, label=t, color=MAST_TYPE_COLORS[t], ls='--')

        ax.set_ylim([-60.0, 25.0])
        ax.legend()
        ax.grid(True)
        ax.set_title('Average curves in segment\n' + segment)
        if j == 5:
            ax.set_xlabel(r'Relative time $t/T_i$')
            
    fig.tight_layout(rect=[0, 0.03, 1, 0.97])